# Reproduce: CNT side (texton matching)

Renders and scores the paper's own first five demo pairs (k=0006, 0009,
0017, 0022, 0053, from `data/pairs_120/pairs.csv`) with every CNT
matching rule (field-linear, primitive-linear, Gaussian, target mean, soft
OT, exact OT). The KID/FID numbers printed partway through use a small
(~235-image) improvised reference, not the paper's frozen 2,819-image one,
so they are not meant to match `paper/numbers.tex`. The final cell instead
checks something that *should* match closely: field-linear's per-pair
placement against `evidence/field_linear_place_1000.csv`, which doesn't
depend on any KID/FID reference at all -- on a real Colab run this passed
at a tight (0.01) tolerance. It is still a DINOv2-based score, though, not
a pixel check, so it depends on whichever DINOv2/timm version is installed
at scoring time; see the last cell's comment for what that means if it
ever starts failing. For the real numbers, use `experiments/cnt_1000.py`
with a GPU and the full pipeline; see the package README.

**Tested on:** Colab, T4 GPU, Python 3.13, detectron2 0.6 built from source
(the install cell takes 5-10 minutes, mostly the detectron2 build).

**Restart-safe by design:** there is exactly one scripted restart, right
after the install cell. Every cell after it re-adds `/content/package` and
`/content/cnt-siga24` to `sys.path` itself and uses absolute paths
throughout, so running "Runtime -> Run cell and below" from the DTD cell
in a freshly-restarted kernel works without re-running the install cell
(everything the install cell produces lives on disk, which a session
restart does not clear -- only Python state is lost).


In [ ]:
#@title Environment setup (no secrets, please)
import os

# Recent huggingface_hub versions auto-detect a Colab runtime and proactively
# look for an HF_TOKEN via Colab's secrets UI -- prompting you for access --
# even for a fully public, anonymous model download (DINOv2's weights need
# no token at all). This is set again in the scoring cell below too (a
# session restart clears process env vars along with everything else in the
# kernel), and rescore.py sets it itself at import time as further defense.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
print("HF_HUB_DISABLE_IMPLICIT_TOKEN=1 -- no Hugging Face secret should be requested below.")


In [ ]:
#@title Get the package
import os, shutil

# CODE_URL: where this package's code is hosted (e.g. the paper's official
# code repository, or an OpenReview-attached archive). Left as a
# placeholder -- fill in before distributing this notebook, or just upload
# package.zip yourself (Colab's file-upload sidebar) before running this
# cell: it lands at /package.zip, not /content/package.zip, so both
# locations are checked below.
CODE_URL = "PLACEHOLDER_CODE_URL"  # e.g. "https://github.com/<org>/<repo>.git"

if not os.path.isdir("/content/package"):
    zip_candidates = ["/package.zip", "/content/package.zip"]
    found_zip = next((p for p in zip_candidates if os.path.exists(p)), None)
    if found_zip:
        print(f"found uploaded package at {found_zip}")
        os.system(f"unzip -qo {found_zip} -d /content/package_unzipped")
        inner = "/content/package_unzipped/package"
        os.makedirs("/content/package", exist_ok=True)
        src_dir = inner if os.path.isdir(inner) else "/content/package_unzipped"
        for name in os.listdir(src_dir):
            shutil.move(os.path.join(src_dir, name), os.path.join("/content/package", name))
    elif not CODE_URL.startswith("PLACEHOLDER"):
        if CODE_URL.endswith(".git"):
            os.system(f"git clone -q {CODE_URL} /content/repo")
            os.system("ln -sfn /content/repo/package /content/package")
        else:
            os.system(f"wget -q {CODE_URL} -O /content/package.zip")
            os.system("unzip -qo /content/package.zip -d /content/package")
    else:
        raise RuntimeError(
            "No package found at /package.zip or /content/package.zip, and CODE_URL is still "
            "the placeholder. Either upload package.zip via the Colab file sidebar, or set CODE_URL above."
        )
else:
    print("/content/package already present (this is expected after a restart -- disk persists).")

%cd /content/package
!ls


In [ ]:
#@title Install CNT exactly as the paper notebooks did
import os, sys, importlib.util
os.system('pip install -q timm lpips einops gdown')
os.system('cd /content && rm -rf cnt-siga24 && git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24 && cd cnt-siga24 && git checkout -q 07269302610c92ce38e2dbbe31c492fd09e1c73e')
os.makedirs('/content/cnt-siga24/ckpts', exist_ok=True)
os.system('gdown -q 1kIGSZlI6O_rEssw9IYaZyBA0wycD8JBG -O /content/cnt-siga24/ckpts/finetuned-model256.ckpt')
CNT = '/content/cnt-siga24'; os.chdir(CNT); sys.path.insert(0, CNT)
try:
    from src.inference.core import TexAutoEncoderInference, load_image
except Exception as e:
    print('installing missing deps:', type(e).__name__)
    os.system('pip install -q -r requirements.in; pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
if importlib.util.find_spec('detectron2') is None:
    os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system('pip install -q -U timm "hydra-core>=1.3" "omegaconf>=2.3" "antlr4-python3-runtime==4.9.3"')
os.system('bash /content/package/install/scoring.sh')
os.chdir('/content')
print('ckpt:', os.path.getsize('/content/cnt-siga24/ckpts/finetuned-model256.ckpt')//10**6, 'MB |',
      'detectron2:', importlib.util.find_spec('detectron2') is not None)
print('>>> Now: Runtime → Restart session, then click the DTD cell → Run cell and below.')


## Restart the session now

`Runtime -> Restart session`, then click the next cell (the DTD cell) and
use `Runtime -> Run cell and below`.

Nothing below this point depends on any Python variable from the install
cell above -- everything it produced (the package, CNT, the checkpoint,
every pip-installed package) is on disk, which a session restart does not
clear. Every cell from here on re-establishes its own `sys.path` and uses
absolute paths, so you do not need to re-run the install cell.


In [ ]:
#@title Get the paper's 5 demo pairs (from data/pairs_120/pairs.csv)
import sys, os, json
sys.path.insert(0, "/content/package")
sys.path.insert(0, "/content/cnt-siga24")

from pathlib import Path
from texton_matching.data import download_dtd, prepare_demo_endpoints

# These are the paper's own first five demo pairs (data/pairs_120/pairs.csv,
# k=0006/0009/0017/0022/0053 -- bubbly-smeared, dotted-paisley, knitted-zigzagged,
# veined-zigzagged, banded-pleated), NOT freshly resampled ones: the evidence CSVs
# this notebook checks against were computed for exactly these pairs, and a
# fresh random draw (even with a fixed seed) would draw different pairs and
# make the final PASS/FAIL check meaningless.
#
# prepare_demo_endpoints is the SAME function colab/reproduce_tm.ipynb calls,
# so both notebooks render from byte-identical endpoint images at the same
# path (/content/rendered/A|B/<k>.png) -- an earlier version had each
# notebook build its own copies at different paths, which broke a
# cross-check expecting one shared layout.
try:
    dtd_images = download_dtd("/content/dtd")
except Exception as e:
    print(f"DTD download failed ({type(e).__name__}: {e}); "
          f"prepare_demo_endpoints will fall back to the shipped evidence/demo_pairs/ PNGs")
    dtd_images = None

demo = prepare_demo_endpoints("/content/package", "/content/rendered", dtd_images)
import pandas as pd
print(pd.DataFrame(demo).to_string(index=False))

manifest = [dict(k=row["k"]) for row in demo]
Path("/content/rendered/manifest.json").write_text(json.dumps(dict(pairs=manifest)))
print("wrote /content/rendered/manifest.json for", [m["k"] for m in manifest])


In [ ]:
#@title Render every CNT rule for the 5 demo pairs
import sys, json
sys.path.insert(0, "/content/package")
sys.path.insert(0, "/content/cnt-siga24")

from pathlib import Path
from texton_matching.cnt import CNTModel
from texton_matching import matching

# CNTModel chdirs into the CNT repo root while building the model (its
# Mask2Former config paths are resolved relative to cwd) and restores the
# caller's cwd afterward -- you don't need to manage that yourself.
cnt = CNTModel("/content/cnt-siga24", device="cuda")
manifest = json.loads(Path("/content/rendered/manifest.json").read_text())["pairs"]

# Same rules rescore.py's table1_cnt_1000 table scores below: NOT ot_sym
# (removed from the paper) and NOT tm (that's Texture Mixer's own output,
# a different notebook/script entirely).
methods = {
    "pixel_linear": "field_linear",
    "texton_linear": "primitive_linear",
    "gaussian_w2": "gaussian",
    "ot": "soft_ot",
    "exact": "exact_ot",
    "target_mean": "target_mean",
}
out = Path("/content/rendered/tm_scale1000")
for m in list(methods) + ["recon_A", "recon_B"]:
    (out / m).mkdir(parents=True, exist_ok=True)

for pair in manifest:
    k = pair["k"]  # a 4-digit string, e.g. "0006" -- matches evidence/*.csv's keys
    A_path = Path(f"/content/rendered/A/{k}.png")
    B_path = Path(f"/content/rendered/B/{k}.png")
    imgA, imgB = cnt.load_image(A_path), cnt.load_image(B_path)
    d = cnt.build_pair(imgA, imgB)
    cnt.to_pil(cnt.field_linear(d, 0.0)).save(out / "recon_A" / f"{k}.png")
    d_swap = cnt.build_pair(imgB, imgB)
    cnt.to_pil(cnt.field_linear(d_swap, 0.0)).save(out / "recon_B" / f"{k}.png")
    for method, rule in methods.items():
        decoded = matching.interpolate(cnt, d, rule, 0.5, seed=int(k))
        cnt.to_pil(decoded).save(out / method / f"{k}.png")
    print(f"pair {k}: rendered {list(methods)}")

print("done rendering.")


In [ ]:
#@title Score them (quick reference; see the markdown note at the top)
import os, sys
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")  # re-set after the restart; see top of notebook
sys.path.insert(0, "/content/package")
sys.path.insert(0, "/content/cnt-siga24")

from pathlib import Path
from texton_matching.data import download_dtd
from rescore import Extractor, build_or_load_reference

# A quick ~235-image reference (NOT the paper's frozen 2,819-image one) so
# this smoke test runs fast. per_cat=5 trims the canonical seed=2819,
# 60-per-category sample down for speed; this only affects the printed
# KID/FID's absolute scale, not the placement check in the last cell.
dtd_images = download_dtd("/content/dtd")  # no-op if already downloaded by the DTD cell above
ext = Extractor(device="cuda")
ref_inc, ref_dino = build_or_load_reference(
    Path("/content/scored"), dtd_images, ext, seed=2819, per_cat=5
)


In [ ]:
#@title Run rescore.py end to end and print the rows
import sys
sys.path.insert(0, "/content/package")

# Absolute paths throughout -- don't rely on the notebook's cwd matching
# /content/package at this point.
!python /content/package/rescore.py --images /content/rendered --data-root /content/scored \
    --dtd-root /content/dtd/images --table table1_cnt_1000 --device cuda

import pandas as pd
df = pd.read_csv("/content/scored/matching_ablation/final128_n1000.csv")
print(df.to_string(index=False))


In [ ]:
#@title PASS/FAIL: field-linear placement against the shipped evidence CSV
import sys
sys.path.insert(0, "/content/package")
sys.path.insert(0, "/content/cnt-siga24")

import json
import pandas as pd
from pathlib import Path
from PIL import Image
from rescore import balance_place

# Placement ("place" = d(mid,A)/(d(mid,A)+d(mid,B)) in DINOv2 space) does
# NOT depend on the KID/FID reference at all -- it's purely a distance
# between three images we already rendered. So, unlike the KID/FID numbers
# in the previous cell, THIS should match the archive's own per-pair values
# closely (same CNT commit, same checkpoint, same DTD source images and
# load128 preprocessing as the paper). Because the DTD cell above used the
# paper's own pair ids (not freshly-sampled ones), every one of the 5 demo
# pairs is guaranteed to be in evidence/field_linear_place_1000.csv -- no
# fallback branch is needed here.
evidence = pd.read_csv("/content/package/evidence/field_linear_place_1000.csv")
evidence["k"] = evidence["k"].apply(lambda x: f"{int(x):04d}")
evidence = evidence.set_index("k")

manifest = json.loads(Path("/content/rendered/manifest.json").read_text())["pairs"]
check_ks = [p["k"] for p in manifest]
missing = [k for k in check_ks if k not in evidence.index]
assert not missing, (
    f"pair id(s) {missing} are not in evidence/field_linear_place_1000.csv -- this should be "
    f"impossible given the DTD cell uses evidence/pairs_demo.csv's own pair ids; if you see this, "
    f"the two evidence files have drifted out of sync."
)
print(f"checking {len(check_ks)} pair(s): {check_ks}")

render_dir = Path("/content/rendered/tm_scale1000")
recon_a_dir, recon_b_dir, mid_dir = render_dir / "recon_A", render_dir / "recon_B", render_dir / "pixel_linear"

ext_dino = ext.dino_feats  # reuse the Extractor built two cells up
mid_dino = ext_dino([Image.open(mid_dir / f"{k}.png") for k in check_ks])
a_dino = ext_dino([Image.open(recon_a_dir / f"{k}.png") for k in check_ks])
b_dino = ext_dino([Image.open(recon_b_dir / f"{k}.png") for k in check_ks])
computed_place = balance_place(mid_dino, a_dino, b_dino)

TOL = 0.01  # placement is a deterministic function of three images given a fixed DINOv2
# feature extractor, so this should be tight, not KID-loose -- but it is still a DINOv2-based
# score, not a pixel check: "tight" means "tight for whatever DINOv2/timm version is installed
# right now," not "exactly bit-reproducible forever." A future DINOv2/timm release could shift
# this the way it shifted the TM side's placement check by 0.05-0.065 (see reproduce_tm.ipynb's
# last cell) even with an unchanged render; if this cell starts failing, check that before
# assuming CNT itself regressed.
rows = []
all_pass = True
for i, k in enumerate(check_ks):
    expected = float(evidence.loc[k, "place"])
    got = float(computed_place[i])
    diff = got - expected
    ok = abs(diff) <= TOL
    all_pass &= ok
    rows.append(dict(k=k, expected=expected, computed=got, diff=diff, status="PASS" if ok else "FAIL"))

result_df = pd.DataFrame(rows)
print(result_df.to_string(index=False))
print()
print("OVERALL:", "PASS" if all_pass else "FAIL", f"(tolerance {TOL})")
if not all_pass:
    raise AssertionError("field-linear placement did not reproduce the archive's per-pair values -- "
                          "see the diffs above; this points at an environment or CNT-version difference, "
                          "not a KID/reference issue (placement doesn't use one).")
